# TextGrad prompt optimization for the text-to-SQL assistant

Optimizes the assistant's **system prompt** with [TextGrad](https://github.com/zou-group/textgrad)
(autograd through *textual* gradients), reusing the project's existing building blocks:

- the same dataset and the same seeded, group-aware **train/val/test split** as the GEPA harness
  (`split_dataset`), so results are comparable;
- the existing **FLEX-style LLM-as-Judge** (`build_sql_judge_scorer`) which executes both the
  generated and reference SQL against DuckDB and returns a pass/fail **plus a written rationale**.

Per `specs/textgrad-mlflow-integration/spec.md`, three model roles are configured independently:

| role | what it does |
|------|--------------|
| **task** | answers the question using the prompt being optimized |
| **optimizer / backward** | reads the textual gradient and proposes prompt edits |
| **judge** | the shared SQL judge — scores a candidate and returns the rationale |

The judge's **rationale is the feedback signal** TextGrad backpropagates into the system prompt
(spec FR11); its pass/fail outcome produces the comparable before/after quality numbers.

This is a *minimal* notebook: it prints before/after quality and the final prompt; it does **not**
write MLflow runs or register prompts.

### Two design notes

1. **Only the instruction block is optimized; the schema is fixed context.** The DB schema is large
   and we never want the optimizer to touch it, so it lives *outside* the optimizable `Variable` and
   is injected into the task model's system prompt at call time (`SchemaInjectingEngine`). This also
   keeps the backward/optimizer prompts small.
2. **Endpoint choice matters for the backward step.** `alias-eve` is served by **blablador**, not
   kisski. In testing, blablador reliably **dropped the connection on TextGrad's backward calls**
   (`RemoteProtocolError: Server disconnected`), while **kisski** completed them. So the defaults
   below use `glm-4.7` on **kisski** for all three roles. Edit `*_MODEL` / `*_ENDPOINT` freely
   (valid endpoints are the keys of `ENDPOINTS`: `kisski`, `blablador`); to use `alias-eve`, set its
   endpoint to `blablador` — but expect the optimizer/backward role to be flaky there.

`textgrad` is a project dependency (`uv add textgrad`).

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import textgrad as tg
from dotenv import load_dotenv
from openai import OpenAI
from textgrad.engine.local_model_openai_api import ChatExternalClient

from Evaluating_prompt_optimization_techniques_for_water_management_LLM_assistant_with_RAG.text2sql.core import (
    SYSTEM_PROMPT_TEMPLATE,
    USER_PROMPT_TEMPLATE,
    clean_sql,
    format_schema_for_prompt,
    load_schema,
)
from experiments.text2sql.harness import (
    ENDPOINTS,
    build_sql_judge_scorer,
    load_dataset,
    render_system_prompt,
)
from experiments.text2sql.sampler import split_dataset

/home/shpilevo/work/Evaluating-prompt-optimization-techniques-for-water-management-analysis-assistant-with-RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

Everything experimenter-tunable lives here. `EPOCHS` / `BATCH_SIZE` / `MAX_STEPS_PER_EPOCH` set the
optimization effort (spec FR10); start small to keep API cost and runtime down.

In [ ]:
REPO = Path.cwd().parent  # notebooks/ -> repo root

QUESTIONS_PATH = REPO / "data/text2sql/deflated_75_sqls_prod.json"
SCHEMA_PATH = REPO / "src/Evaluating_prompt_optimization_techniques_for_water_management_LLM_assistant_with_RAG/tenants/green_roof/sensordata.py"
DB_PATH = REPO / "data/water.duckdb"
EMBED_CACHE = QUESTIONS_PATH.with_name("question_embeddings.npz")
USE_PROD_QUESTIONS = True  # deflated_75_sqls_prod.json -> use the 'prod_question' phrasing

# Three model roles (model_string, endpoint). Defaults: glm-4.7 on kisski for all roles
# (reliable backward calls). To use the project's alias-eve, set its endpoint to "blablador"
# -- but note blablador dropped TextGrad's backward connections in testing.
TASK_MODEL, TASK_ENDPOINT = "glm-4.7", "kisski"
OPT_MODEL, OPT_ENDPOINT = "glm-4.7", "kisski"  # optimizer / backward engine
JUDGE_MODEL, JUDGE_ENDPOINT = "glm-4.7", "kisski"

# Effort (spec FR10). NOTE: a backward step makes several sequential LLM calls and is the
# slow part. These defaults are a small smoke test; raise BATCH_SIZE / set
# MAX_STEPS_PER_EPOCH=None / EVAL_LIMIT=None for a fuller run.
SEED = 42
EPOCHS = 1
BATCH_SIZE = 2
MAX_STEPS_PER_EPOCH = 2  # cap steps/epoch; set to None for a full pass over the train split
# val/test are scored several times (baseline + per epoch + final). Subsample for a quick
# smoke run; set to None to score the full splits (needed for comparability with GEPA).
EVAL_LIMIT = 8

# litellm wants the 'openai/<name>' style; the judge is built on litellm.
JUDGE_MODEL_LITELLM = f"openai/{JUDGE_MODEL}"

## Setup: schema, data split, shared judge

In [ ]:
load_dotenv(REPO / ".env")

schema_text = format_schema_for_prompt(load_schema(str(SCHEMA_PATH)))
data = load_dataset(str(QUESTIONS_PATH), use_prod_questions=USE_PROD_QUESTIONS)
train_set, val_set, test_set = split_dataset(data, SEED, cache_path=EMBED_CACHE)
print(f"split (seed={SEED}): train={len(train_set)} val={len(val_set)} test={len(test_set)}")

# Shared FLEX-style judge: executes both SQLs against DuckDB (EQ/NEQ branch) and
# returns a Feedback with .value (pass/fail) and .rationale.
judge_scorer = build_sql_judge_scorer(
    JUDGE_MODEL_LITELLM, JUDGE_ENDPOINT, schema_text, str(DB_PATH)
)

## TextGrad engines

`ChatExternalClient` wraps an OpenAI-compatible client pointed at the project endpoint — the
documented way to target a custom `base_url`, and it lets each role use a different endpoint.

The **task** engine is a `SchemaInjectingEngine`: it appends the fixed DB schema to whatever
(optimizable) instruction prompt it is given, so the model always sees the schema while the schema
itself stays out of the optimizable `Variable`. The **backward/optimizer** engine is a plain
`ChatExternalClient` and only ever sees the small instruction text + gradients.

In [ ]:
class SchemaInjectingEngine(ChatExternalClient):
    """Task engine that appends the fixed DB schema to whatever (optimizable) system
    prompt it receives, so the schema reaches the model on every call but never lives
    inside the optimizable Variable. This keeps the schema fixed AND keeps the
    backward/optimizer prompts small (a full schema in the gradient prompt is large
    and was observed to drop the endpoint connection)."""

    def __init__(self, *args, schema_text: str, **kwargs):
        super().__init__(*args, **kwargs)
        self._schema_text = schema_text

    def generate(self, content, system_prompt=None, **kwargs):
        base = system_prompt if system_prompt is not None else self.system_prompt
        return super().generate(
            content, system_prompt=render_system_prompt(base, self._schema_text), **kwargs
        )


def make_client(endpoint: str) -> OpenAI:
    base_var, key_var = ENDPOINTS[endpoint]
    return OpenAI(base_url=os.environ[base_var], api_key=os.environ[key_var])


# Task engine sees the schema (injected per call); the backward/optimizer engine does not.
task_engine = SchemaInjectingEngine(
    client=make_client(TASK_ENDPOINT), model_string=TASK_MODEL, schema_text=schema_text
)
backward_engine = ChatExternalClient(client=make_client(OPT_ENDPOINT), model_string=OPT_MODEL)
# The backward engine computes the textual gradients and drives the optimizer.
tg.set_backward_engine(backward_engine, override=True)

## Optimizable system prompt

Only the **instruction block** (everything before the `Schema:` section of `SYSTEM_PROMPT_TEMPLATE`)
is the optimizable `tg.Variable`. The schema is fixed context injected by the task engine, so it
never enters the gradient/optimizer prompts. Two TGD **constraints** keep the strict output rules
and stop the optimizer from re-introducing the schema.

In [ ]:
# Optimize ONLY the instruction block; the schema is fixed context the task engine injects.
instructions = SYSTEM_PROMPT_TEMPLATE.split("Schema:")[0].rstrip()

system_prompt = tg.Variable(
    instructions,
    requires_grad=True,
    role_description=(
        "instruction block of the system prompt for a DuckDB text-to-SQL assistant; "
        "the database schema is supplied separately and must not be repeated here"
    ),
)
task_model = tg.BlackboxLLM(task_engine, system_prompt)

optimizer = tg.TGD(
    parameters=[system_prompt],
    constraints=[
        "Keep the strict output rules: return ONLY a single DuckDB-dialect SQL query, no prose or markdown.",
        "Do not paste or invent the database schema, table names, or column names; the schema is provided separately.",
    ],
)

## Judge-as-loss bridge

`sql_judge_loss` runs the task model, judges its SQL with the shared judge, then wraps the judge's
**rationale** in a `TextLoss` applied to the model output. Backprop therefore grounds the textual
gradient on the judge's rationale and pushes it into the system prompt. The instruction tells the
backward engine to *use* the rationale, not re-judge.

In [ ]:
JUDGE_LOSS_TEMPLATE = (
    "An expert SQL judge has evaluated the SQL produced above for the user's question.\n"
    "Verdict: {verdict}.\n"
    "Judge rationale: {rationale}\n\n"
    "Do NOT re-evaluate or solve the task yourself. Using ONLY the judge's rationale, explain what\n"
    "about the instructions that generated this SQL should change so future SQL is correct. If the\n"
    "verdict is CORRECT, briefly affirm what worked so it is preserved."
)


def judge_one(question: str, ref_sql: str, sql: str):
    """Call the shared judge directly; returns the Feedback (.value, .rationale)."""
    return judge_scorer(
        inputs={"question": question},
        outputs={"sql": sql},
        expectations={"sql": ref_sql, "argilla_link": ""},
    )


def sql_judge_loss(question: str, ref_sql: str):
    """Forward + judge, returning (loss_variable, passed_bool)."""
    q_var = tg.Variable(
        USER_PROMPT_TEMPLATE.format(question=question),
        requires_grad=False,
        role_description="natural-language question for the text-to-SQL assistant",
    )
    response = task_model(q_var)  # graph: system_prompt -> response
    fb = judge_one(question, ref_sql, clean_sql(response.value))
    verdict = "CORRECT" if fb.value else "INCORRECT"
    eval_instruction = tg.Variable(
        JUDGE_LOSS_TEMPLATE.format(verdict=verdict, rationale=fb.rationale),
        requires_grad=False,
        role_description="expert judge evaluation of the generated SQL",
    )
    loss = tg.TextLoss(eval_instruction)(response)
    return loss, bool(fb.value)


def eval_split(records, limit=EVAL_LIMIT) -> float:
    # Mean judge pass-rate over a split (no gradients). `limit` subsamples the first
    # `limit` records for a faster smoke run; None scores the whole split.
    if limit is not None:
        records = records[:limit]
    passed = []
    for rec in records:
        q = rec["inputs"]["question"]
        ref = rec["expectations"]["sql"]
        resp = task_model(
            tg.Variable(
                USER_PROMPT_TEMPLATE.format(question=q),
                requires_grad=False,
                role_description="natural-language question",
            )
        )
        fb = judge_one(q, ref, clean_sql(resp.value))
        passed.append(1.0 if fb.value else 0.0)
    return float(np.mean(passed)) if passed else 0.0

## Baseline (before optimization)

In [ ]:
val_before = eval_split(val_set)
test_before = eval_split(test_set)
print(f"baseline   val={val_before:.2%}  test={test_before:.2%}")

# Track the best-on-validation prompt (spec FR12 / SC2a): keep the best, not the last.
best_prompt = system_prompt.get_value()
best_val = val_before

## Training loop (epochs + validation revert)

PyTorch-style: for each minibatch, zero grads, accumulate per-example judge losses, backward, step.
After each epoch, evaluate on validation and **revert** the prompt if it did not improve.

In [ ]:
def batches(records, size):
    for i in range(0, len(records), size):
        yield records[i : i + size]


history = []
for epoch in range(EPOCHS):
    for step, batch in enumerate(batches(train_set, BATCH_SIZE)):
        if MAX_STEPS_PER_EPOCH is not None and step >= MAX_STEPS_PER_EPOCH:
            break
        optimizer.zero_grad()
        losses, hits = [], []
        for rec in batch:
            loss, passed = sql_judge_loss(
                rec["inputs"]["question"], rec["expectations"]["sql"]
            )
            losses.append(loss)
            hits.append(passed)
        tg.sum(losses).backward()
        optimizer.step()
        print(f"epoch {epoch} step {step}: batch train pass-rate {np.mean(hits):.2%}")

    val_acc = eval_split(val_set)
    if val_acc < best_val:
        print(f"  epoch {epoch}: val {val_acc:.2%} < best {best_val:.2%} -> revert")
        system_prompt.set_value(best_prompt)
    else:
        print(f"  epoch {epoch}: val {val_acc:.2%} >= best {best_val:.2%} -> keep")
        best_val = val_acc
        best_prompt = system_prompt.get_value()
    history.append({"epoch": epoch, "val_acc": val_acc, "best_val": best_val})

# Make sure the live prompt is the best-on-val one before final scoring.
system_prompt.set_value(best_prompt)
pd.DataFrame(history)

## Results (after optimization)

In [ ]:
val_after = eval_split(val_set)
test_after = eval_split(test_set)

summary = pd.DataFrame(
    {
        "split": ["val", "test"],
        "before": [val_before, test_before],
        "after": [val_after, test_after],
        "delta": [val_after - val_before, test_after - test_before],
    }
)
print(summary.to_string(index=False))
if val_after <= val_before and test_after <= test_before:
    print("\nNote: optimization did not beat the baseline on either split (spec EC2).")

In [ ]:
# The optimized *instruction block*. At runtime the task engine appends the fixed schema,
# so the full system prompt the model sees is render_system_prompt(value, schema_text).
print("=== optimized instruction block ===\n")
print(system_prompt.get_value())